# 🥉 Bronze Layer — Development Notebook

Use this notebook to test and develop the Bronze layer logic locally before pushing to dev.

**Flow:** Raw CSV → Schema enforcement + metadata → BigQuery (bronze_stock_data)

In [ ]:
# ── Setup: Install dependencies (run once) ──
# !pip install pyspark pandas

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
import pandas as pd

# Create local Spark session (no GCP required)
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("BronzeLayer_Dev")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## 1. Create Sample Data
Simulate raw CSV data for local testing (no API key needed).

In [ ]:
# Create sample data to mimic Alpha Vantage CSV output
sample_data = [
    ("2024-01-02", 185.50, 187.20, 184.80, 186.90, 186.90, 50000000, 0.0, 1.0, "AAPL"),
    ("2024-01-03", 186.50, 188.00, 185.00, 185.50, 185.50, 48000000, 0.0, 1.0, "AAPL"),
    ("2024-01-04", 182.00, 183.50, 181.00, 182.70, 182.70, 55000000, 0.0, 1.0, "AAPL"),
    ("2024-01-05", 183.00, 184.00, 182.50, 183.80, 183.80, 42000000, 0.0, 1.0, "AAPL"),
    ("2024-01-02", 140.50, 142.00, 139.80, 141.20, 141.20, 30000000, 0.0, 1.0, "GOOGL"),
    ("2024-01-03", 141.00, 143.50, 140.50, 142.80, 142.80, 28000000, 0.0, 1.0, "GOOGL"),
    ("2024-01-04", 139.00, 140.00, 138.00, 139.50, 139.50, 35000000, 0.0, 1.0, "GOOGL"),
    ("2024-01-05", 140.00, 141.50, 139.50, 141.00, 141.00, 25000000, 0.0, 1.0, "GOOGL"),
    # Bad rows for testing quality filters later
    (None,         100.00, 101.00, 99.00,  100.50, 100.50, 10000000, 0.0, 1.0, "MSFT"),     # null date
    ("2024-01-02", -5.00,  101.00, 99.00,  100.50, 100.50, 10000000, 0.0, 1.0, "MSFT"),     # negative open
    ("2024-01-02", 100.00, 101.00, 99.00,  100.50, 100.50, 10000000, 0.0, 1.0, "MSFT"),     # valid
]

columns = ["date", "open", "high", "low", "close", "adjusted_close", "volume", "dividend_amount", "split_coefficient", "symbol"]

raw_df = spark.createDataFrame(sample_data, columns)
print(f"Sample rows: {raw_df.count()}")
raw_df.show(truncate=False)

## 2. Bronze Logic — Schema Enforcement + Metadata

In [ ]:
# ── Add ingestion metadata (same as bronze_layer.py) ──
bronze_df = (
    raw_df
    .withColumn("load_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("sample_data"))       # lit() for local testing
    .withColumn("load_date", F.current_date())
)

print(f"Bronze rows: {bronze_df.count()}")
bronze_df.printSchema()
bronze_df.show(truncate=False)

## 3. Validate Schema
Check that the schema matches what Terraform expects.

In [ ]:
expected_columns = [
    "date", "open", "high", "low", "close", "adjusted_close",
    "volume", "dividend_amount", "split_coefficient", "symbol",
    "load_timestamp", "source_file", "load_date"
]

actual_columns = bronze_df.columns

missing = set(expected_columns) - set(actual_columns)
extra = set(actual_columns) - set(expected_columns)

print(f"✅ All expected columns present: {len(missing) == 0}")
if missing: print(f"❌ Missing: {missing}")
if extra:   print(f"⚠️  Extra:   {extra}")

## 4. Save Locally (for Silver notebook testing)

In [ ]:
# Save as parquet for Silver notebook to consume
bronze_df.write.mode("overwrite").parquet("data/test/bronze_output")
print("✅ Bronze output saved to data/test/bronze_output/")

In [ ]:
# spark.stop()  # Uncomment when done